
# Phase 1: Data Pipeline Validation
This notebook validates the healthy T1w axial slice pipeline before training any diffusion model. It discovers subjects, applies preprocessing, verifies normalization and filters, tests a Windows-safe DataLoader, and saves validation logs and figures.


In [10]:

# Environment setup and imports
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import seaborn as sns
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import torch.nn.functional as F

DATA_ROOT = Path(r"C:\Users\ayush\Downloads\Brats\brain_only")
LOG_DIR = Path.cwd() / "logs"
FIG_DIR = LOG_DIR / "figures"
LOG_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_theme(style="darkgrid")

print('DATA_ROOT:', DATA_ROOT)
print('Logs directory:', LOG_DIR)
print('Figure directory:', FIG_DIR)

try:
    assert DATA_ROOT.exists(), f"DATA_ROOT does not exist: {DATA_ROOT}"
except AssertionError as exc:
    raise FileNotFoundError(str(exc))


DATA_ROOT: C:\Users\ayush\Downloads\Brats\brain_only
Logs directory: c:\Users\ayush\Downloads\Brats\logs
Figure directory: c:\Users\ayush\Downloads\Brats\logs\figures



## Section A — Data discovery
Scan the `brain_only` root for subject folders containing `t1_brain.nii.gz`. This ensures the dataset is discovered without any index file.


In [11]:

def find_subjects(root_path):
    valid_subjects = []
    missing_subjects = []
    for child in sorted(root_path.iterdir()):
        if not child.is_dir():
            continue
        candidate = child / "t1_brain.nii.gz"
        if candidate.exists():
            valid_subjects.append(child)
        else:
            missing_subjects.append(child)
    return valid_subjects, missing_subjects

subjects, missing = find_subjects(DATA_ROOT)
print(f'Total subjects discovered: {len(subjects)}')
print('First 20 subject IDs:')
print([s.name for s in subjects[:20]])
if missing:
    print(f'Folders missing t1_brain.nii.gz ({len(missing)}):')
    print([p.name for p in missing])
else:
    print('No missing subject files detected.')

if not subjects:
    raise RuntimeError('No subject directories with t1_brain.nii.gz were found under DATA_ROOT.')

random.seed(42)
subjects_sorted = sorted(subjects, key=lambda x: x.name)
random.shuffle(subjects_sorted)
n = len(subjects_sorted)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
train_subjects = subjects_sorted[:n_train]
val_subjects = subjects_sorted[n_train:n_train + n_val]
test_subjects = subjects_sorted[n_train + n_val:]

print('Subject counts:')
print(f'  train: {len(train_subjects)}')
print(f'  val:   {len(val_subjects)}')
print(f'  test:  {len(test_subjects)}')
print('First 5 train subject IDs:', [p.name for p in train_subjects[:5]])
print('First 5 val subject IDs:', [p.name for p in val_subjects[:5]])
print('First 5 test subject IDs:', [p.name for p in test_subjects[:5]])


Total subjects discovered: 210
First 20 subject IDs:
['2APWTMIH5Z', '2CO3ODNYCK', '2SRPTDXMEG', '2WG4BHKT3S', '2XK7DWSUKA', '2ZND6GHU5B', '2ZTLIWL4JN', '3BJIJ47YV7', '3BR673ZMTE', '3N5ZZH7O27', '3YSMZNZ7TT', '3YXTPSCFSO', '4EEX4LEWG7', '4IJW45J4TC', '4IQRPEHVCL', '4MQTBXPXVI', '4WMIZI7U4S', '53A3SL6ORH', '53MGT2KYFL', '57RCXT5FGU']
No missing subject files detected.
Subject counts:
  train: 168
  val:   21
  test:  21
First 5 train subject IDs: ['6YUYK6TBPK', 'YWFQHYJBII', 'I4ZUUTIOR4', 'G2JTKTHEBX', 'XEEPP5GLYB']
First 5 val subject IDs: ['W5JILIQZFI', '75JHHM6NOZ', 'T4EGCFKCES', 'G7M4SG7XPG', 'WXNYFAL2QP']
First 5 test subject IDs: ['BUX5SW6Y5D', 'BL7MFVN4BT', '5H32UAAHLO', '3BJIJ47YV7', '3BR673ZMTE']



## Section B — Single volume inspection
Load one training subject and print volume-level metadata before preprocessing. Then apply each preprocessing step individually and verify the results.


In [12]:

def preprocess_volume(nii_path):
    try:
        img = nib.load(str(nii_path))
    except Exception as exc:
        raise RuntimeError(f'Failed to load NIfTI volume {nii_path}: {exc}')

    vol = img.get_fdata(dtype=np.float32)
    if vol.ndim != 3:
        raise ValueError(f'Expected 3D volume, got shape {vol.shape} from {nii_path}')

    nonzero = vol[vol != 0]
    if nonzero.size == 0:
        return []

    p05, p995 = np.percentile(nonzero, [0.5, 99.5])
    if p995 <= p05:
        normalized = np.clip(vol, 0.0, 1.0).astype(np.float32)
    else:
        normalized = np.clip((vol - p05) / (p995 - p05), 0.0, 1.0).astype(np.float32)

    z_count = normalized.shape[2]
    z_start = int(np.floor(z_count * 0.15))
    z_end = int(np.ceil(z_count * 0.85))
    z_end = min(z_end, z_count)
    z_indices = np.arange(z_start, z_end, dtype=int)

    valid_slices = []
    for z in z_indices:
        slice_2d = normalized[:, :, z]
        fill_ratio = float(np.count_nonzero(slice_2d) / slice_2d.size)
        if fill_ratio < 0.05:
            continue
        if slice_2d.shape != (256, 256):
            tensor_slice = torch.from_numpy(slice_2d[None, None].astype(np.float32))
            tensor_slice = F.interpolate(tensor_slice, size=(256, 256), mode='bilinear', align_corners=False)
            slice_2d = tensor_slice.squeeze().cpu().numpy().astype(np.float32)
        valid_slices.append(slice_2d)
    return valid_slices

def preprocess_volume_with_indices(nii_path):
    try:
        img = nib.load(str(nii_path))
    except Exception as exc:
        raise RuntimeError(f'Failed to load NIfTI volume {nii_path}: {exc}')
    vol = img.get_fdata(dtype=np.float32)
    if vol.ndim != 3:
        raise ValueError(f'Expected 3D volume, got shape {vol.shape} from {nii_path}')
    nonzero = vol[vol != 0]
    if nonzero.size == 0:
        return [], []
    p05, p995 = np.percentile(nonzero, [0.5, 99.5])
    if p995 <= p05:
        normalized = np.clip(vol, 0.0, 1.0).astype(np.float32)
    else:
        normalized = np.clip((vol - p05) / (p995 - p05), 0.0, 1.0).astype(np.float32)
    z_count = normalized.shape[2]
    z_start = int(np.floor(z_count * 0.15))
    z_end = int(np.ceil(z_count * 0.85))
    z_end = min(z_end, z_count)
    z_indices = np.arange(z_start, z_end, dtype=int)
    retained_slices = []
    retained_z = []
    for z in z_indices:
        slice_2d = normalized[:, :, z]
        fill_ratio = float(np.count_nonzero(slice_2d) / slice_2d.size)
        if fill_ratio < 0.05:
            continue
        if slice_2d.shape != (256, 256):
            tensor_slice = torch.from_numpy(slice_2d[None, None].astype(np.float32))
            tensor_slice = F.interpolate(tensor_slice, size=(256, 256), mode='bilinear', align_corners=False)
            slice_2d = tensor_slice.squeeze().cpu().numpy().astype(np.float32)
        retained_slices.append(slice_2d)
        retained_z.append(int(z))
    return retained_slices, retained_z

example_subject = train_subjects[0]
example_path = example_subject / "t1_brain.nii.gz"
print('Inspecting subject:', example_subject.name)
try:
    example_img = nib.load(str(example_path))
except Exception as exc:
    raise RuntimeError(f'Could not load example subject volume: {exc}')
example_data = example_img.get_fdata(dtype=np.float32)
print('Raw volume shape:', example_data.shape)
spacing = np.sqrt((example_img.affine[:3, :3] ** 2).sum(axis=0))
print('Voxel spacing (mm):', spacing)
print('Dtype:', example_data.dtype)
print('Raw min/max:', float(example_data.min()), float(example_data.max()))

nonzero = example_data[example_data != 0]
if nonzero.size > 0:
    p05, p995 = np.percentile(nonzero, [0.5, 99.5])
    normalized_example = np.clip((example_data - p05) / max(p995 - p05, 1e-6), 0.0, 1.0).astype(np.float32)
else:
    normalized_example = example_data.astype(np.float32)
print('After percentile normalization range:', float(normalized_example.min()), float(normalized_example.max()))
print('After percentile normalization mean/std:', float(normalized_example.mean()), float(normalized_example.std()))

z_count = normalized_example.shape[2]
z_start = int(np.floor(z_count * 0.15))
z_end = int(np.ceil(z_count * 0.85))
z_end = min(z_end, z_count)
z_indices = np.arange(z_start, z_end, dtype=int)
print('Z-range filter keeps indices from', z_start, 'to', z_end - 1)
print('Candidate z slices before fill filter:', len(z_indices))

valid_after_fill = []
discarded = 0
for z in z_indices:
    slice_2d = normalized_example[:, :, z]
    fill_ratio = float(np.count_nonzero(slice_2d) / slice_2d.size)
    if fill_ratio < 0.05:
        discarded += 1
        continue
    valid_after_fill.append(slice_2d)
print('Slices discarded by fill filter:', discarded)
print('Slices retained after all filters:', len(valid_after_fill))
print('Example retained slice shape:', valid_after_fill[0].shape if valid_after_fill else 'none')


Inspecting subject: 6YUYK6TBPK
Raw volume shape: (204, 256, 25)
Voxel spacing (mm): [0.85937501 0.859375   5.99999986]
Dtype: float32
Raw min/max: 0.0 697.0
After percentile normalization range: 0.0 1.0
After percentile normalization mean/std: 0.16260282695293427 0.30808770656585693
Z-range filter keeps indices from 3 to 21
Candidate z slices before fill filter: 19
Slices discarded by fill filter: 0
Slices retained after all filters: 19
Example retained slice shape: (204, 256)



### Visualize valid slices from a single example subject
Display up to 16 evenly spaced retained slices from the selected subject to verify appearance and indexing.


In [13]:

example_slices, example_z = preprocess_volume_with_indices(example_path)
n_vis = min(16, len(example_slices))
if n_vis == 0:
    raise RuntimeError('No valid slices found for the example subject.')
indices = np.linspace(0, len(example_slices) - 1, n_vis, dtype=int)
fig, axs = plt.subplots(4, 4, figsize=(16, 16), constrained_layout=True)
for plot_idx, dataset_idx in enumerate(indices):
    row = plot_idx // 4
    col = plot_idx % 4
    ax = axs[row, col]
    slice_data = example_slices[dataset_idx]
    ax.imshow(slice_data, cmap='gray', vmin=0.0, vmax=1.0)
    ax.set_title(f'z={example_z[dataset_idx]}')
    ax.axis('off')
fig.suptitle('Example valid slices from one training subject', fontsize=16)
fig.savefig(FIG_DIR / 'example_subject_valid_slices.png', dpi=150)
plt.close(fig)



## Section C — Population-level checks
Apply preprocessing to all training subjects and collect per-subject slice statistics, then verify distribution and clipping artifacts.


In [14]:

subject_stats = []
flagged_zero_subjects = []
for subject_path in tqdm(train_subjects, desc='Preprocessing train subjects'):
    subject_file = subject_path / "t1_brain.nii.gz"
    slices, _ = preprocess_volume_with_indices(subject_file)
    if len(slices) == 0:
        flagged_zero_subjects.append(subject_path.name)
        subject_stats.append({'subject_id': subject_path.name, 'n_slices_retained': 0, 'mean': None, 'std': None})
        continue
    all_pixels = np.concatenate([sl.ravel() for sl in slices])
    subject_stats.append({
        'subject_id': subject_path.name,
        'n_slices_retained': len(slices),
        'mean': float(all_pixels.mean()),
        'std': float(all_pixels.std()),
    })

counts = np.array([s['n_slices_retained'] for s in subject_stats if s['n_slices_retained'] > 0], dtype=np.float32)
means = np.array([s['mean'] for s in subject_stats if s['mean'] is not None], dtype=np.float32)
stds = np.array([s['std'] for s in subject_stats if s['std'] is not None], dtype=np.float32)

def summarize(arr):
    return {
        'mean': float(arr.mean()) if arr.size else None,
        'std': float(arr.std()) if arr.size else None,
        'min': float(arr.min()) if arr.size else None,
        'max': float(arr.max()) if arr.size else None,
    }

print('Slices retained per subject summary:')
print(summarize(counts))
print('Pixel mean per subject summary:')
print(summarize(means))
print('Pixel std per subject summary:')
print(summarize(stds))
if flagged_zero_subjects:
    print('Subjects with zero valid slices:', flagged_zero_subjects)
else:
    print('No subjects were flagged with zero valid slices.')


Preprocessing train subjects:   0%|          | 0/168 [00:00<?, ?it/s]

Slices retained per subject summary:
{'mean': 68.55952453613281, 'std': 82.73701477050781, 'min': 14.0, 'max': 278.0}
Pixel mean per subject summary:
{'mean': 0.16145120561122894, 'std': 0.03154066205024719, 'min': 0.0784677267074585, 'max': 0.260651558637619}
Pixel std per subject summary:
{'mean': 0.29545578360557556, 'std': 0.032354503870010376, 'min': 0.17236168682575226, 'max': 0.36565178632736206}
No subjects were flagged with zero valid slices.



### Plot slices-per-subject histogram
Visualize how many retained slices each training subject contributes.


In [ ]:

counts = [s['n_slices_retained'] for s in subject_stats]
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(counts, bins=20, kde=False, ax=ax, color='tab:blue')
ax.set_title('Training slices per subject distribution')
ax.set_xlabel('Number of retained slices')
ax.set_ylabel('Number of subjects')
fig.savefig(FIG_DIR / 'slices_per_subject_histogram.png', dpi=150)
plt.close(fig)



### Pool random slices and inspect pixel distribution
Sample up to 2000 slices from the training split and check the pooled histogram for clipping artifacts.


In [ ]:

pooled_slices = []
for subject_path in train_subjects:
    subject_file = subject_path / "t1_brain.nii.gz"
    slices, _ = preprocess_volume_with_indices(subject_file)
    pooled_slices.extend(slices)
    if len(pooled_slices) >= 2000:
        break
pooled_slices = pooled_slices[:2000]
pooled_pixels = np.concatenate([sl.ravel() for sl in pooled_slices]) if pooled_slices else np.array([], dtype=np.float32)
print('Pooled slice count:', len(pooled_slices))
if pooled_pixels.size > 0:
    print('Pooled pixel min/max:', float(pooled_pixels.min()), float(pooled_pixels.max()))
    print('Zero boundary count:', int((pooled_pixels == 0.0).sum()))
    print('One boundary count:', int((pooled_pixels == 1.0).sum()))
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(pooled_pixels, bins=100, kde=False, ax=ax, color='tab:purple')
ax.set_title('Pooled pixel value histogram for up to 2000 training slices')
ax.set_xlim(0.0, 1.0)
ax.set_xlabel('Pixel value')
fig.savefig(FIG_DIR / 'pooled_pixel_histogram.png', dpi=150)
plt.close(fig)



### Z-position distribution for random subjects
Plot retained axial indices for 10 random training subjects to verify the 15-85% filter.


In [ ]:

selected = random.sample(train_subjects, min(10, len(train_subjects)))
fig, ax = plt.subplots(figsize=(12, 6))
for subject_path in selected:
    subject_file = subject_path / "t1_brain.nii.gz"
    _, z_positions = preprocess_volume_with_indices(subject_file)
    if z_positions:
        ax.scatter(z_positions, [subject_path.name] * len(z_positions), s=10)
ax.set_title('Retained axial slice positions for 10 training subjects')
ax.set_xlabel('Slice index')
ax.set_ylabel('Subject ID')
fig.savefig(FIG_DIR / 'retained_z_positions.png', dpi=150)
plt.close(fig)



### Full NaN / Inf / out-of-range check
Scan all training subjects after preprocessing and report any invalid values.


In [ ]:

invalid_reports = []
for subject_path in tqdm(train_subjects, desc='Integrity check train subjects'):
    subject_file = subject_path / "t1_brain.nii.gz"
    slices, _ = preprocess_volume_with_indices(subject_file)
    for z_idx, sl in enumerate(slices):
        if not np.isfinite(sl).all() or sl.min() < 0.0 or sl.max() > 1.0:
            invalid_reports.append({
                'subject_id': subject_path.name,
                'slice_index': z_idx,
                'min': float(sl.min()),
                'max': float(sl.max()),
                'finite': bool(np.isfinite(sl).all()),
            })
            break
if invalid_reports:
    print('Found invalid slices after preprocessing:')
    for report in invalid_reports[:10]:
        print(report)
else:
    print('No NaN, Inf, or out-of-range values found in the training split.')



## Section D — DataLoader test
Define a Windows-safe dataset that loads a random valid slice per subject and tests batch loading and augmentation.


In [ ]:

class MRRATESliceDataset(Dataset):
    def __init__(self, subject_paths, transform=None):
        self.subject_paths = list(subject_paths)
        self.transform = transform if transform is not None else T.RandomHorizontalFlip(p=0.5)
        if not self.subject_paths:
            raise ValueError('No subjects provided to MRRATESliceDataset.')

    def __len__(self):
        return len(self.subject_paths)

    def __getitem__(self, idx):
        subject_path = self.subject_paths[idx]
        volume_path = subject_path / "t1_brain.nii.gz"
        slices, _ = preprocess_volume_with_indices(volume_path)
        if not slices:
            raise RuntimeError(f'No valid slices for subject {subject_path.name}')
        chosen = random.choice(slices)
        tensor = torch.from_numpy(chosen.astype(np.float32)).unsqueeze(0)
        if self.transform is not None:
            tensor = self.transform(tensor)
        return tensor

try:
    dataset = MRRATESliceDataset(train_subjects)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)
    print('DataLoader created with', len(dataset), 'subjects.')
except Exception as exc:
    raise RuntimeError(f'Failed to initialize DataLoader: {exc}')



### Time 50 DataLoader batches
Measure throughput for 50 batches with batch size 8 and report slices per second.


In [ ]:

num_batches = min(50, len(dataloader))
start = time.perf_counter()
processed = 0
for i, batch in enumerate(dataloader):
    if i >= num_batches:
        break
    processed += batch.shape[0]
duration = time.perf_counter() - start
throughput = float(processed) / max(duration, 1e-6)
print(f'Processed {processed} slices in {duration:.3f}s ({throughput:.1f} slices/sec)')



### Visualize one batch and confirm shape
Display the first batch of 8 slices and confirm their tensor shape matches the target.


In [ ]:

first_batch = next(iter(dataloader))
print('Batch tensor shape:', tuple(first_batch.shape))
assert first_batch.shape == (8, 1, 256, 256), 'Batch shape is not (8,1,256,256)'

fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
axes = axes.flatten()
for i in range(8):
    axes[i].imshow(first_batch[i, 0].numpy(), cmap='gray', vmin=0.0, vmax=1.0)
    axes[i].set_title(f'Batch {i}')
    axes[i].axis('off')
fig.suptitle('One batch of 8 augmented training slices')
fig.savefig(FIG_DIR / 'one_batch_visualization.png', dpi=150)
plt.close(fig)



### Augmentation effect on the same slice
Load the same slice repeatedly and display the results to verify horizontal flip augmentation is applied randomly.


In [ ]:

same_subject = train_subjects[0]
slices, _ = preprocess_volume_with_indices(same_subject / "t1_brain.nii.gz")
if not slices:
    raise RuntimeError('No valid slices available for augmentation check.')
base_slice = slices[0]
transform = T.RandomHorizontalFlip(p=0.5)
augmented = []
for _ in range(8):
    tensor = torch.from_numpy(base_slice.astype(np.float32)).unsqueeze(0)
    augmented.append(transform(tensor).squeeze(0).numpy())
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
axes = axes.flatten()
for i in range(8):
    axes[i].imshow(augmented[i], cmap='gray', vmin=0.0, vmax=1.0)
    axes[i].set_title(f'Augment {i}')
    axes[i].axis('off')
fig.suptitle('Augmentation effect on the same slice')
fig.savefig(FIG_DIR / 'augmentation_effect.png', dpi=150)
plt.close(fig)



## Section E — Summary and logs
Save data validation summary to `logs/data_validation.json` for later reference.


In [ ]:

split_summary = {
    'total_subjects': len(subjects),
    'train_subjects': len(train_subjects),
    'val_subjects': len(val_subjects),
    'test_subjects': len(test_subjects),
    'total_slices_train': sum(s['n_slices_retained'] for s in subject_stats),
    'train_pixel_mean_summary': None,
    'train_pixel_std_summary': None,
    'zero_valid_slice_subjects': flagged_zero_subjects,
}
if means.size:
    split_summary['train_pixel_mean_summary'] = {
        'mean': float(means.mean()),
        'std': float(means.std()),
        'min': float(means.min()),
        'max': float(means.max()),
    }
if stds.size:
    split_summary['train_pixel_std_summary'] = {
        'mean': float(stds.mean()),
        'std': float(stds.std()),
        'min': float(stds.min()),
        'max': float(stds.max()),
    }
with open(LOG_DIR / 'data_validation.json', 'w', encoding='utf-8') as f:
    json.dump(split_summary, f, indent=2)
print('Saved validation summary to', LOG_DIR / 'data_validation.json')
